In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [15]:
df = pd.read_csv('./datasets/structures.csv')


df_100 = df.head(100)

In [16]:
df_100.head()

,compound_id,smiles
0,1,C(=O)(C(C(C(F)(F)F)(F)F)(F)F)O
1,2,C(=O)(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)O
2,3,C(=O)(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)O
3,4,C(=O)(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(...
4,5,C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...


In [17]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_100,
    smiles_col="smiles",
    id_col="compound_id",
    keywords="PM7 PRECISE GEO-OK",
    workdir="mopac_jobs",
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6,
)

In [18]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [19]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir="mopac_jobs",
    overwrite=True,
    timeout=900,
    max_workers=6,
)

In [20]:
print(results[results['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [21]:
print(results[results['arc_file'].isna()])#.to_csv('failed_runs.csv')

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [26]:
#results

In [23]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir="mopac_jobs",
    overwrite=True,
    max_workers=6,
)

In [24]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [25]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\mopac_jobs")
target_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)